In [21]:
import pandas as pd
import pubchempy as pcp
import time
import requests

def get_pubchem_compound_data(smiles):
    """
    Retrieve comprehensive information for a given SMILES string from PubChem
    """
    try:
         
        compounds = pcp.get_compounds(smiles, 'smiles')
        
        if not compounds:
            return {
                'SMILES': smiles,
                'PubChem_CID': 'Not Found',
                'Molecular_Formula': 'N/A',
                'Molecular_Weight': 'N/A',
                'Canonical_SMILES': 'N/A',
                'Synonyms': 'N/A',
                'GC-MS_Data': 'No compound found'
            }
        
         
        compound = compounds[0]
        
         
        try:
            cid = compound.cid
            rest_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/property/MolecularFormula,MolecularWeight,CanonicalSMILES,IUPACName/JSON'
            response = requests.get(rest_url)
            additional_data = response.json()
            
             
            properties = additional_data.get('PropertyTable', {}).get('Properties', [{}])[0]
            
             
            gc_ms_url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/description/JSON'
            desc_response = requests.get(gc_ms_url)
            descriptions = desc_response.json()
            
             
            gc_ms_descriptions = []
            if 'InformationList' in descriptions:
                for desc in descriptions['InformationList']['Information']:
                    description = desc.get('Description', '')
                    if 'GC-MS' in description or 'mass spectrometry' in description.lower():
                        gc_ms_descriptions.append(description)
            
            return {
                'SMILES': smiles,
                'PubChem_CID': cid,
                'Molecular_Formula': properties.get('MolecularFormula', 'N/A'),
                'Molecular_Weight': properties.get('MolecularWeight', 'N/A'),
                'Canonical_SMILES': properties.get('CanonicalSMILES', 'N/A'),
                'IUPAC_Name': properties.get('IUPACName', 'N/A'),
                'Synonyms': ', '.join(compound.synonyms[:5]) if compound.synonyms else 'N/A',
                'GC-MS_Data': ' | '.join(gc_ms_descriptions) if gc_ms_descriptions else 'No specific GC-MS data found',
            }
        
        except Exception as api_error:
            return {
                'SMILES': smiles,
                'PubChem_CID': compound.cid,
                'Molecular_Formula': compound.molecular_formula,
                'Molecular_Weight': compound.molecular_weight,
                'Canonical_SMILES': compound.canonical_smiles,
                'Synonyms': ', '.join(compound.synonyms[:5]) if compound.synonyms else 'N/A',
                'GC-MS_Data': f'Error fetching additional data: {str(api_error)}'
            }
    
    except Exception as e:
        return {
            'SMILES': smiles,
            'PubChem_CID': 'Error',
            'GC-MS_Data': str(e)
        }

def extract_pubchem_data_from_csv(input_csv, output_csv):
    """
    Extract PubChem information from SMILES in input CSV
    """
     
    df = pd.read_csv(input_csv)
    
     
    compound_results = []
    
     
    smiles_list = df['SMILE'].tolist()
    
     
    for smiles in smiles_list:
        print(f"Processing SMILES: {smiles}")
        result = get_pubchem_compound_data(smiles)
        compound_results.append(result)
        
         
        time.sleep(1)
    
     
    output_df = pd.DataFrame(compound_results)
    
     
    output_df.to_csv(output_csv, index=False)
    
    print(f"PubChem compound information saved to {output_csv}")

 


 
input_csv = 'data/VGAE/filtered_generated_smiles.csv'
output_csv = 'data/VGAE/pubchem_gcms_results.csv'
extract_pubchem_data_from_csv(input_csv, output_csv)
 
 

 


Processing SMILES: CCCC
Processing SMILES: C=C
Processing SMILES: CC(C)C
PubChem compound information saved to data/VGAE/pubchem_gcms_results.csv


In [1]:
import os
import pandas as pd
import requests
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, RDConfig
import sys
import numpy as np
from urllib.parse import quote

 
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
 
try:
    import sascorer
    SA_SCORE_AVAILABLE = True
except ImportError:
    print("Warning: SA Score module not found in RDKit. Will not calculate SA scores.")
    SA_SCORE_AVAILABLE = False

 
def validate_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, None
    return True, mol

 
def calculate_sa_score(mol):
    if SA_SCORE_AVAILABLE:
        try:
            sa_score = sascorer.calculateScore(mol)
             
             
            if sa_score <= 3:
                difficulty = "Easy"
            elif sa_score <= 6:
                difficulty = "Moderate"
            else:
                difficulty = "Difficult"
            
            return {
                'sa_score': sa_score,
                'synthesis_difficulty': difficulty,
                'synthesis_confidence': max(0, min(100, 100 - (sa_score * 10))),   
            }
        except Exception as e:
            print(f"Error calculating SA score: {e}")
    return {
        'sa_score': None,
        'synthesis_difficulty': "Unknown",
        'synthesis_confidence': None,
    }

 
def get_pubchem_info(smiles):
    try:
         
        smiles_encoded = quote(smiles)
        url_cid = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles_encoded}/cids/JSON"
        response = requests.get(url_cid)
        
        if response.status_code != 200:
            return {
                'pubchem_cid': None,
                'pubchem_direct_reference': None,
                'pubchem_reference_count': 0,
                'pubchem_references': '',
                'experimental_properties': False
            }
        
        data = response.json()
        if 'IdentifierList' not in data or 'CID' not in data['IdentifierList']:
            return {
                'pubchem_cid': None,
                'pubchem_direct_reference': None,
                'pubchem_reference_count': 0,
                'pubchem_references': '',
                'experimental_properties': False
            }
        
        cid = data['IdentifierList']['CID'][0]
        
         
        compound_info = {
            'pubchem_cid': cid,
            'pubchem_direct_reference': f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}",
        }
        
         
        url_refs = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/PubMedID/JSON"
        refs_response = requests.get(url_refs)
        
        pubmed_ids = []
        if refs_response.status_code == 200:
            refs_data = refs_response.json()
            if 'InformationList' in refs_data and 'Information' in refs_data['InformationList']:
                info = refs_data['InformationList']['Information'][0]
                if 'PubMedID' in info:
                    pubmed_ids = info['PubMedID']
        
         
        references = []
        for pmid in pubmed_ids[:5]:   
            references.append(f"PubMed: https://pubmed.ncbi.nlm.nih.gov/{pmid}/")
        
        compound_info['pubchem_reference_count'] = len(pubmed_ids)
        compound_info['pubchem_references'] = '; '.join(references) if references else ''
        
         
        url_props = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/MolecularFormula,MolecularWeight,XLogP,HBondDonorCount,HBondAcceptorCount,RotatableBondCount/JSON"
        props_response = requests.get(url_props)
        
        if props_response.status_code == 200:
            props_data = props_response.json()
            if 'PropertyTable' in props_data and 'Properties' in props_data['PropertyTable']:
                properties = props_data['PropertyTable']['Properties'][0]
                 
                compound_info['experimental_properties'] = True
                
                 
                if 'MolecularFormula' in properties:
                    compound_info['molecular_formula'] = properties['MolecularFormula']
                if 'MolecularWeight' in properties:
                    compound_info['pubchem_molecular_weight'] = properties['MolecularWeight']
                if 'XLogP' in properties:
                    compound_info['pubchem_xlogp'] = properties['XLogP']
        else:
            compound_info['experimental_properties'] = False
        
        return compound_info
    
    except Exception as e:
        print(f"Error in PubChem API: {e}")
        return {
            'pubchem_cid': None,
            'pubchem_direct_reference': None,
            'pubchem_reference_count': 0,
            'pubchem_references': '',
            'experimental_properties': False
        }

 
def get_chembl_data(smiles):
    try:
         
        base_url = "https://www.ebi.ac.uk/chembl/api/data"
        encoded_smiles = quote(smiles)
        url = f"{base_url}/molecule.json?molecule_structures__canonical_smiles__flexmatch={encoded_smiles}"
        
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data.get('molecules') and len(data['molecules']) > 0:
                molecule = data['molecules'][0]
                chembl_id = molecule.get('molecule_chembl_id')
                
                 
                detail_url = f"{base_url}/molecule/{chembl_id}.json"
                detail_response = requests.get(detail_url)
                
                if detail_response.status_code == 200:
                    detail_data = detail_response.json()
                    
                     
                    activity_url = f"{base_url}/activity.json?molecule_chembl_id={chembl_id}&limit=1"
                    activity_response = requests.get(activity_url)
                    activity_count = 0
                    
                    if activity_response.status_code == 200:
                        activity_data = activity_response.json()
                        activity_count = activity_data.get('page_meta', {}).get('total_count', 0)
                    
                     
                    chembl_reference = f"https://www.ebi.ac.uk/chembl/compound_report_card/{chembl_id}"
                    
                    return {
                        'chembl_id': chembl_id,
                        'chembl_name': detail_data.get('pref_name'),
                        'chembl_activity_count': activity_count,
                        'experimental_validated': activity_count > 0,
                        'chembl_direct_reference': chembl_reference
                    }
        
        return {
            'chembl_id': None,
            'chembl_name': None,
            'chembl_activity_count': 0,
            'experimental_validated': False,
            'chembl_direct_reference': None
        }
    
    except Exception as e:
        print(f"Error fetching ChEMBL data: {e}")
        return {
            'chembl_id': None,
            'chembl_name': None,
            'chembl_activity_count': 0,
            'experimental_validated': False,
            'chembl_direct_reference': None
        }

 
def calculate_qed(mol):
    try:
        qed = Descriptors.qed(mol)
        return qed
    except:
        return None

 
def get_patent_information(smiles):
    try:
         
        encoded_smiles = quote(smiles)
        url = f"https://patents.google.com/api/search?q={encoded_smiles}&page=1"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers)
        patent_count = 0
        patent_references = []
        
        if response.status_code == 200:
            try:
                data = response.json()
                if 'results' in data and 'total' in data['results']:
                    patent_count = data['results']['total']
                
                 
                if 'results' in data and 'cluster' in data['results']:
                    for patent in data['results']['cluster'][:5]:
                        if 'result' in patent and 'patent' in patent['result']:
                            patent_id = patent['result']['patent'].get('publication_number')
                            if patent_id:
                                patent_references.append(f"{patent_id} (Google Patents)")
            except:
                 
                pass
        
         
        try:
            url = f"https://www.surechembl.org/search/structure/resolve?smiles={quote(smiles)}"
            response = requests.get(url, headers={"Accept": "application/json"})
            
            if response.status_code == 200:
                results = response.json()
                if 'total_patents' in results:
                    patent_count = max(patent_count, results.get('total_patents', 0))
                
                 
                if 'patent_numbers' in results and results['patent_numbers']:
                    for patent in results['patent_numbers'][:5]:
                        patent_references.append(f"{patent} (SureChEMBL)")
        except:
            pass
        
        return {
            'patent_count': patent_count,
            'patent_references': '; '.join(set(patent_references)) if patent_references else None
        }
    except Exception as e:
        print(f"Error fetching patent data: {e}")
        return {
            'patent_count': 0,
            'patent_references': None
        }

 
def process_smiles_file(input_file):
     
    if not os.path.exists(input_file):
        print(f"Error: File {input_file} not found")
        return
    
     
    try:
        df = pd.read_csv(input_file)
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return
    
     
    if 'SMILE' not in df.columns:
        print("Error: 'SMILE' column not found in the CSV file")
        return
    
     
    df['is_valid'] = False
    
     
    print(f"Processing {len(df)} molecules...")
    
    for idx, row in df.iterrows():
        if idx % 10 == 0:
            print(f"Processing molecule {idx+1}/{len(df)}...")
            
        smiles = row['SMILE']
        valid, mol = validate_smiles(smiles)
        df.at[idx, 'is_valid'] = valid
        
        if not valid:
            continue
        
         
        df.at[idx, 'molecular_weight'] = Descriptors.MolWt(mol)
        df.at[idx, 'logp'] = Descriptors.MolLogP(mol)
        df.at[idx, 'heavy_atom_count'] = mol.GetNumHeavyAtoms()
        df.at[idx, 'qed_score'] = calculate_qed(mol)
        
         
        sa_data = calculate_sa_score(mol)
        for key, value in sa_data.items():
            df.at[idx, key] = value
        
         
        pubchem_data = get_pubchem_info(smiles)
        for key, value in pubchem_data.items():
            df.at[idx, key] = value
        
         
        chembl_data = get_chembl_data(smiles)
        for key, value in chembl_data.items():
            df.at[idx, key] = value
        
         
        patent_data = get_patent_information(smiles)
        for key, value in patent_data.items():
            df.at[idx, key] = value
        
         
        time.sleep(1)
    
     
    df['experimental_validation'] = 'Unknown'
    
     
    mask_exp_validated = (
        df['is_valid'] & (
            (df['experimental_properties'] == True) |
            (df['experimental_validated'] == True) |
            (df['pubchem_reference_count'] > 0) |
            (df['chembl_activity_count'] > 0)
        )
    )
    df.loc[mask_exp_validated, 'experimental_validation'] = 'Experimentally Validated'
    
     
    synthesizable_mask = df['is_valid'] & (~mask_exp_validated)
    if 'sa_score' in df.columns:
        easy_synth_mask = synthesizable_mask & (df['sa_score'] <= 4.5)
        df.loc[easy_synth_mask, 'experimental_validation'] = 'Readily Synthesizable, Not Validated'
        
        mod_synth_mask = synthesizable_mask & (df['sa_score'] > 4.5) & (df['sa_score'] <= 6)
        df.loc[mod_synth_mask, 'experimental_validation'] = 'Moderately Synthesizable, Not Validated'
    
     
    try:
        mask_valid = df['is_valid'] & (~mask_exp_validated) & (~(easy_synth_mask | mod_synth_mask))
        df.loc[mask_valid, 'experimental_validation'] = 'Valid Structure, Difficult Synthesis'
    except:
         
        mask_valid = df['is_valid'] & (~mask_exp_validated)
        if 'sa_score' in df.columns:
            mask_valid = mask_valid & (df['sa_score'] > 6)
        df.loc[mask_valid, 'experimental_validation'] = 'Valid Structure, Difficult Synthesis'
    
     
    df.loc[~df['is_valid'], 'experimental_validation'] = 'Invalid Structure'
    
     
    df['all_references'] = ''
    for idx, row in df.iterrows():
        refs = []
        
         
        if not pd.isna(row.get('pubchem_direct_reference')) and row.get('pubchem_direct_reference'):
            refs.append(f"PubChem: {row['pubchem_direct_reference']}")
        
         
        if not pd.isna(row.get('pubchem_references')) and row.get('pubchem_references'):
            refs.append(f"Literature: {row['pubchem_references']}")
        
         
        if not pd.isna(row.get('chembl_direct_reference')) and row.get('chembl_direct_reference'):
            refs.append(f"ChEMBL: {row['chembl_direct_reference']}")
        
         
        if not pd.isna(row.get('patent_references')) and row.get('patent_references'):
            refs.append(f"Patents: {row['patent_references']}")
        
        df.at[idx, 'all_references'] = '; '.join(refs)
    
     
    output_file = input_file.replace('.csv', '_experimental_validation.csv')
    df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
    
     
    print("\nSummary:")
    print(f"Total molecules: {len(df)}")
    print(f"Valid molecules: {df['is_valid'].sum()} ({df['is_valid'].mean()*100:.1f}%)")
    print(f"Experimentally validated: {(df['experimental_validation'] == 'Experimentally Validated').sum()}")
    
     
    try:
        print(f"Readily synthesizable: {(df['experimental_validation'] == 'Readily Synthesizable, Not Validated').sum()}")
        print(f"Moderately synthesizable: {(df['experimental_validation'] == 'Moderately Synthesizable, Not Validated').sum()}")
    except:
        pass
    
    print(f"Difficult synthesis: {(df['experimental_validation'] == 'Valid Structure, Difficult Synthesis').sum()}")
    print(f"Invalid structures: {(df['experimental_validation'] == 'Invalid Structure').sum()}")
    
     
    print("\nReference statistics:")
    print(f"Molecules with PubChem references: {(df['pubchem_reference_count'] > 0).sum()}")
    print(f"Molecules with ChEMBL data: {(df['chembl_activity_count'] > 0).sum()}")
    print(f"Molecules with patent references: {(df['patent_count'] > 0).sum()}")
    
    return df

if __name__ == "__main__":
    input_file = "data/VGAE/filtered_generated_smiles.csv"
    results = process_smiles_file(input_file)

Processing 3 molecules...
Processing molecule 1/3...
Results saved to data/VGAE/filtered_generated_smiles_experimental_validation.csv

Summary:
Total molecules: 3
Valid molecules: 3 (100.0%)
Experimentally validated: 3
Readily synthesizable: 0
Moderately synthesizable: 0
Difficult synthesis: 0
Invalid structures: 0

Reference statistics:
Molecules with PubChem references: 3
Molecules with ChEMBL data: 2
Molecules with patent references: 0


In [2]:
import os
import pandas as pd
import requests
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, RDConfig
import sys
import numpy as np
from urllib.parse import quote

 
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
 
try:
    import sascorer
    SA_SCORE_AVAILABLE = True
except ImportError:
    print("Warning: SA Score module not found in RDKit. Will not calculate SA scores.")
    SA_SCORE_AVAILABLE = False

 
def validate_smiles(smiles):
    """
    Validates if a SMILES string represents a chemically valid molecule.
    
    Args:
        smiles (str): The SMILES string to validate
        
    Returns:
        tuple: (is_valid, mol_object) where is_valid is a boolean and mol_object is the RDKit molecule 
               object or None if invalid
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, None
    return True, mol

 
def calculate_sa_score(mol):
    """
    Calculates the Synthetic Accessibility score for a molecule.
    
    SA score ranges from 1 (easy to synthesize) to 10 (difficult to synthesize).
    This function also categorizes molecules by synthesis difficulty and provides 
    a confidence percentage.
    
    Args:
        mol: RDKit molecule object
        
    Returns:
        dict: Dictionary containing sa_score, synthesis_difficulty category, and synthesis_confidence
    """
    if SA_SCORE_AVAILABLE:
        try:
            sa_score = sascorer.calculateScore(mol)
             
             
            if sa_score <= 3:
                difficulty = "Easy"
            elif sa_score <= 6:
                difficulty = "Moderate"
            else:
                difficulty = "Difficult"
            
            return {
                'sa_score': sa_score,
                'synthesis_difficulty': difficulty,
                'synthesis_confidence': max(0, min(100, 100 - (sa_score * 10))),   
            }
        except Exception as e:
            print(f"Error calculating SA score: {e}")
    return {
        'sa_score': None,
        'synthesis_difficulty': "Unknown",
        'synthesis_confidence': None,
    }

def get_pubchem_info(smiles):
    """
    Retrieves comprehensive compound information from PubChem based on SMILES.
    
    This improved function:
    1. Converts SMILES to PubChem CID (Compound ID)
    2. Gets compound basic properties
    3. Gets PubChem direct references
    4. Retrieves PubMed literature references
    5. Gets compound classifications
    6. Retrieves experimental property information
    
    Args:
        smiles (str): SMILES string for the molecule
        
    Returns:
        dict: Dictionary containing PubChem information including CID, references,
              literature citations, and classifications
    """
    default_result = {
        'pubchem_cid': None,
        'pubchem_direct_reference': None,
        'pubchem_reference_count': 0,
        'pubchem_references': '',
        'pubmed_count': 0,
        'pubmed_ids': '',
        'classification_info': '',
        'experimental_properties': False,
        'molecular_formula': None,
        'pubchem_molecular_weight': None,
        'pubchem_xlogp': None
    }
    
    try:
         
        smiles_encoded = quote(smiles)
        url_cid = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles_encoded}/cids/JSON"
        response = requests.get(url_cid)
        
        if response.status_code != 200:
            return default_result
        
        data = response.json()
        if 'IdentifierList' not in data or 'CID' not in data['IdentifierList']:
            return default_result
        
        cid = data['IdentifierList']['CID'][0]
        
         
        compound_info = {
            'pubchem_cid': cid,
            'pubchem_direct_reference': f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}",
            'pubchem_reference_count': 0,   
            'pubchem_references': '',   
            'pubmed_count': 0,
            'pubmed_ids': '',
            'classification_info': '',
            'experimental_properties': False
        }
        
         
        url_datasources = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/SourceName/JSON"
        datasources_response = requests.get(url_datasources)
        
        if datasources_response.status_code == 200:
            try:
                datasources_data = datasources_response.json()
                if 'InformationList' in datasources_data and 'Information' in datasources_data['InformationList']:
                    info = datasources_data['InformationList']['Information'][0]
                    if 'SourceName' in info:
                        sources = info['SourceName']
                        compound_info['pubchem_reference_count'] = len(sources)
                         
                        compound_info['pubchem_references'] = '; '.join(sources[:5])
            except Exception as e:
                print(f"Error processing PubChem data sources: {e}")
        
         
        literature_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/xrefs/PubMedID/JSON"
        lit_response = requests.get(literature_url)
        
        if lit_response.status_code == 200:
            try:
                lit_data = lit_response.json()
                if 'InformationList' in lit_data and 'Information' in lit_data['InformationList']:
                    info = lit_data['InformationList']['Information'][0]
                    if 'PubMedID' in info:
                        pmids = info['PubMedID']
                        compound_info['pubmed_count'] = len(pmids)
                         
                        recent_pmids = pmids[:3]
                        compound_info['pubmed_ids'] = '; '.join([f"PMID:{pmid}" for pmid in recent_pmids])
            except Exception as e:
                print(f"Error processing PubMed IDs: {e}")
        
         
        classification_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON?heading=Classification"
        class_response = requests.get(classification_url)
        
        if class_response.status_code == 200:
            try:
                class_data = class_response.json()
                classifications = []
                
                if 'Record' in class_data and 'Section' in class_data['Record']:
                    sections = class_data['Record']['Section']
                    for section in sections:
                        if section.get('TOCHeading') == 'Classification':
                            if 'Section' in section:
                                for subsection in section['Section']:
                                    if 'Information' in subsection:
                                        for info in subsection['Information']:
                                            if 'StringValue' in info:
                                                classifications.append(f"{subsection.get('TOCHeading', 'Class')}: {info['StringValue']}")
                
                if classifications:
                    compound_info['classification_info'] = '; '.join(classifications[:3])
            except Exception as e:
                print(f"Error processing classification data: {e}")
        
         
        url_props = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/MolecularFormula,MolecularWeight,XLogP,HBondDonorCount,HBondAcceptorCount,RotatableBondCount/JSON"
        props_response = requests.get(url_props)
        
        if props_response.status_code == 200:
            props_data = props_response.json()
            if 'PropertyTable' in props_data and 'Properties' in props_data['PropertyTable']:
                properties = props_data['PropertyTable']['Properties'][0]
                 
                compound_info['experimental_properties'] = True
                
                 
                if 'MolecularFormula' in properties:
                    compound_info['molecular_formula'] = properties['MolecularFormula']
                if 'MolecularWeight' in properties:
                    compound_info['pubchem_molecular_weight'] = properties['MolecularWeight']
                if 'XLogP' in properties:
                    compound_info['pubchem_xlogp'] = properties['XLogP']
        else:
            compound_info['experimental_properties'] = False
        gc_ms=  f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid} 
        compound_info['Mass_Spectrometry_info']=gc_ms
        return compound_info
    
    except Exception as e:
        print(f"Error in PubChem API: {e}")
        return default_result
 
def get_chembl_data(smiles):
    """
    Retrieves information about a molecule from the ChEMBL database.
    
    ChEMBL contains curated bioactivity data of drug-like molecules and
    provides information on whether a compound has been experimentally tested.
    
    Args:
        smiles (str): SMILES string for the molecule
        
    Returns:
        dict: Dictionary containing ChEMBL ID, name, activity count,
              experimental validation status, and direct reference
    """
    try:
         
        base_url = "https://www.ebi.ac.uk/chembl/api/data"
        encoded_smiles = quote(smiles)
        url = f"{base_url}/molecule.json?molecule_structures__canonical_smiles__flexmatch={encoded_smiles}"
        
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data.get('molecules') and len(data['molecules']) > 0:
                molecule = data['molecules'][0]
                chembl_id = molecule.get('molecule_chembl_id')
                
                 
                detail_url = f"{base_url}/molecule/{chembl_id}.json"
                detail_response = requests.get(detail_url)
                
                if detail_response.status_code == 200:
                    detail_data = detail_response.json()
                    
                     
                    activity_url = f"{base_url}/activity.json?molecule_chembl_id={chembl_id}&limit=1"
                    activity_response = requests.get(activity_url)
                    activity_count = 0
                    
                    if activity_response.status_code == 200:
                        activity_data = activity_response.json()
                        activity_count = activity_data.get('page_meta', {}).get('total_count', 0)
                    
                     
                    chembl_reference = f"https://www.ebi.ac.uk/chembl/compound_report_card/{chembl_id}"
                    
                    return {
                        'chembl_id': chembl_id,
                        'chembl_name': detail_data.get('pref_name'),
                        'chembl_activity_count': activity_count,
                        'experimental_validated': activity_count > 0,
                        'chembl_direct_reference': chembl_reference
                    }
        
        return {
            'chembl_id': None,
            'chembl_name': None,
            'chembl_activity_count': 0,
            'experimental_validated': False,
            'chembl_direct_reference': None
        }
    
    except Exception as e:
        print(f"Error fetching ChEMBL data: {e}")
        return {
            'chembl_id': None,
            'chembl_name': None,
            'chembl_activity_count': 0,
            'experimental_validated': False,
            'chembl_direct_reference': None
        }

 
def calculate_qed(mol):
    """
    Calculates the Quantitative Estimate of Drug-likeness (QED) for a molecule.
    
    QED ranges from 0 (not drug-like) to 1 (very drug-like). This is a composite
    measure based on multiple properties that define drug-likeness.
    
    Args:
        mol: RDKit molecule object
        
    Returns:
        float: QED value between 0 and 1, or None if calculation fails
    """
    try:
        qed = Descriptors.qed(mol)
        return qed
    except:
        return None

 
def get_patent_information(smiles):
    """
    Searches for patents related to a molecule in patent databases.
    
    This function tries two approaches:
    1. Google Patents API
    2. SureChEMBL database
    
    Args:
        smiles (str): SMILES string for the molecule
        
    Returns:
        dict: Dictionary containing patent_count and patent_references
    """
    try:
         
        encoded_smiles = quote(smiles)
        url = f"https://patents.google.com/api/search?q={encoded_smiles}&page=1"
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers)
        patent_count = 0
        patent_references = []
        
        if response.status_code == 200:
            try:
                data = response.json()
                if 'results' in data and 'total' in data['results']:
                    patent_count = data['results']['total']
                
                 
                if 'results' in data and 'cluster' in data['results']:
                    for patent in data['results']['cluster'][:5]:
                        if 'result' in patent and 'patent' in patent['result']:
                            patent_id = patent['result']['patent'].get('publication_number')
                            if patent_id:
                                patent_references.append(f"{patent_id} (Google Patents)")
            except:
                 
                pass
        
         
        try:
            url = f"https://www.surechembl.org/search/structure/resolve?smiles={quote(smiles)}"
            response = requests.get(url, headers={"Accept": "application/json"})
            
            if response.status_code == 200:
                results = response.json()
                if 'total_patents' in results:
                    patent_count = max(patent_count, results.get('total_patents', 0))
                
                 
                if 'patent_numbers' in results and results['patent_numbers']:
                    for patent in results['patent_numbers'][:5]:
                        patent_references.append(f"{patent} (SureChEMBL)")
        except:
            pass
        
        return {
            'patent_count': patent_count,
            'patent_references': '; '.join(set(patent_references)) if patent_references else None
        }
    except Exception as e:
        print(f"Error fetching patent data: {e}")
        return {
            'patent_count': 0,
            'patent_references': None
        }

 
def process_smiles_file(input_file):
    """
    Processes a CSV file containing SMILES strings to validate and enrich with additional data.
    
    The function:
    1. Validates each SMILES
    2. Calculates molecular properties
    3. Determines synthesizability
    4. Checks databases (PubChem, ChEMBL) for existing records
    5. Looks for patent information
    6. Categorizes molecules based on experimental validation
    
    Args:
        input_file (str): Path to CSV file containing SMILES column
        
    Returns:
        DataFrame: Processed dataframe with additional columns
    """
     
    if not os.path.exists(input_file):
        print(f"Error: File {input_file} not found")
        return
    
     
    try:
        print(f"Reading file: {input_file}")
        df = pd.read_csv(input_file)
        print(f"Successfully loaded file with {len(df)} rows and {len(df.columns)} columns")
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return
    
     
    if 'SMILE' not in df.columns:
        print("Error: 'SMILE' column not found in the CSV file")
        print(f"Available columns: {', '.join(df.columns)}")
        return
    
     
    df['is_valid'] = False
    
     
    print(f"Processing {len(df)} molecules...")
    
    for idx, row in df.iterrows():
        if idx % 10 == 0:
            print(f"Processing molecule {idx+1}/{len(df)}...")
            
        smiles = row['SMILE']
        valid, mol = validate_smiles(smiles)
        df.at[idx, 'is_valid'] = valid
        
        if not valid:
            print(f"  Molecule {idx+1} has an invalid SMILES: {smiles}")
            continue
        
         
        print(f"  Calculating properties for molecule {idx+1}")
        df.at[idx, 'molecular_weight'] = Descriptors.MolWt(mol)
        df.at[idx, 'logp'] = Descriptors.MolLogP(mol)
        df.at[idx, 'heavy_atom_count'] = mol.GetNumHeavyAtoms()
        
         
        qed_value = calculate_qed(mol)
        print(f"  Molecule {idx+1} QED (not saved to CSV): {qed_value}")
        
         
        print(f"  Calculating synthesizability for molecule {idx+1}")
        sa_data = calculate_sa_score(mol)
        for key, value in sa_data.items():
            df.at[idx, key] = value
        
         
        print(f"  Fetching PubChem data for molecule {idx+1}")
        pubchem_data = get_pubchem_info(smiles)
        for key, value in pubchem_data.items():
            df.at[idx, key] = value
        
         
        print(f"  Fetching ChEMBL data for molecule {idx+1}")
        chembl_data = get_chembl_data(smiles)
        for key, value in chembl_data.items():
            df.at[idx, key] = value
        
         
        print(f"  Searching for patents for molecule {idx+1}")
        patent_data = get_patent_information(smiles)
        for key, value in patent_data.items():
            df.at[idx, key] = value
        
         
        print(f"  Summary for molecule {idx+1}:")
        print(f"    Molecular weight: {df.at[idx, 'molecular_weight']:.2f}")
        print(f"    LogP: {df.at[idx, 'logp']:.2f}")
        if 'sa_score' in df.columns and df.at[idx, 'sa_score'] is not None:
            print(f"    Synthetic accessibility score: {df.at[idx, 'sa_score']:.2f} ({df.at[idx, 'synthesis_difficulty']})")
        if df.at[idx, 'pubchem_cid'] is not None:
            print(f"    Found in PubChem (CID: {df.at[idx, 'pubchem_cid']})")
        if df.at[idx, 'chembl_id'] is not None:
            print(f"    Found in ChEMBL (ID: {df.at[idx, 'chembl_id']})")
        if df.at[idx, 'patent_count'] > 0:
            print(f"    Found {df.at[idx, 'patent_count']} patents")
        
         
        time.sleep(1)
    
     
    print("\nDetermining experimental validation status for all molecules...")
    df['experimental_validation'] = 'Unknown'
    
     
    mask_exp_validated = (
        df['is_valid'] & (
            (df['experimental_properties'] == True) |
            (df['experimental_validated'] == True) |
            (df['pubchem_reference_count'] > 0) |
            (df['chembl_activity_count'] > 0)
        )
    )
    df.loc[mask_exp_validated, 'experimental_validation'] = 'Experimentally Validated'
    print(f"  Found {mask_exp_validated.sum()} experimentally validated molecules")
    
     
    synthesizable_mask = df['is_valid'] & (~mask_exp_validated)
    if 'sa_score' in df.columns:
        easy_synth_mask = synthesizable_mask & (df['sa_score'] <= 4.5)
        df.loc[easy_synth_mask, 'experimental_validation'] = 'Readily Synthesizable, Not Validated'
        print(f"  Found {easy_synth_mask.sum()} readily synthesizable molecules (not validated)")
        
        mod_synth_mask = synthesizable_mask & (df['sa_score'] > 4.5) & (df['sa_score'] <= 6)
        df.loc[mod_synth_mask, 'experimental_validation'] = 'Moderately Synthesizable, Not Validated'
        print(f"  Found {mod_synth_mask.sum()} moderately synthesizable molecules (not validated)")
    
     
    try:
        mask_valid = df['is_valid'] & (~mask_exp_validated) & (~(easy_synth_mask | mod_synth_mask))
        df.loc[mask_valid, 'experimental_validation'] = 'Valid Structure, Difficult Synthesis'
        print(f"  Found {mask_valid.sum()} molecules that are difficult to synthesize")
    except:
         
        mask_valid = df['is_valid'] & (~mask_exp_validated)
        if 'sa_score' in df.columns:
            mask_valid = mask_valid & (df['sa_score'] > 6)
        df.loc[mask_valid, 'experimental_validation'] = 'Valid Structure, Difficult Synthesis'
        print(f"  Found {mask_valid.sum()} molecules that are difficult to synthesize (fallback calculation)")
    
     
    invalid_count = (~df['is_valid']).sum()
    df.loc[~df['is_valid'], 'experimental_validation'] = 'Invalid Structure'
    print(f"  Found {invalid_count} invalid structures")
    
     
    print("\nCompiling references for all molecules...")
    df['all_references'] = ''
    for idx, row in df.iterrows():
        refs = []
        
         
        if not pd.isna(row.get('pubchem_direct_reference')) and row.get('pubchem_direct_reference'):
            refs.append(f"PubChem: {row['pubchem_direct_reference']}")
        
         
        if not pd.isna(row.get('pubchem_references')) and row.get('pubchem_references'):
            refs.append(f"PubChem Sources: {row['pubchem_references']}")
        
         
        if not pd.isna(row.get('chembl_direct_reference')) and row.get('chembl_direct_reference'):
            refs.append(f"ChEMBL: {row['chembl_direct_reference']}")
        
         
        if not pd.isna(row.get('patent_references')) and row.get('patent_references'):
            refs.append(f"Patents: {row['patent_references']}")
        
        df.at[idx, 'all_references'] = '; '.join(refs)
    
     
    output_file = input_file.replace('.csv', '_experimental_validation.csv')
    print(f"\nSaving results to {output_file}...")
    
     
    if 'qed_score' in df.columns:
        print("Removing QED scores from the output CSV as requested")
        df = df.drop(columns=['qed_score'])
    
    df.to_csv(output_file, index=False)
    print(f"Results successfully saved to {output_file}")
    
     
    print("\n===== SUMMARY =====")
    print(f"Total molecules: {len(df)}")
    print(f"Valid molecules: {df['is_valid'].sum()} ({df['is_valid'].mean()*100:.1f}%)")
    print(f"Experimentally validated: {(df['experimental_validation'] == 'Experimentally Validated').sum()}")
    
     
    try:
        print(f"Readily synthesizable: {(df['experimental_validation'] == 'Readily Synthesizable, Not Validated').sum()}")
        print(f"Moderately synthesizable: {(df['experimental_validation'] == 'Moderately Synthesizable, Not Validated').sum()}")
    except:
        pass
    
    print(f"Difficult synthesis: {(df['experimental_validation'] == 'Valid Structure, Difficult Synthesis').sum()}")
    print(f"Invalid structures: {(df['experimental_validation'] == 'Invalid Structure').sum()}")
    
     
    print("\nReference statistics:")
    print(f"Molecules with PubChem references: {(df['pubchem_reference_count'] > 0).sum()}")
    print(f"Molecules with ChEMBL data: {(df['chembl_activity_count'] > 0).sum()}")
    print(f"Molecules with patent references: {(df['patent_count'] > 0).sum()}")
    
     
    print("\nMolecular property statistics (valid molecules only):")
    valid_mols = df[df['is_valid']]
    if not valid_mols.empty:
        print(f"Average molecular weight: {valid_mols['molecular_weight'].mean():.2f}")
        print(f"Average LogP: {valid_mols['logp'].mean():.2f}")
        if 'sa_score' in valid_mols.columns:
            print(f"Average SA score: {valid_mols['sa_score'].mean():.2f}")
    
    return df

if __name__ == "__main__":
    input_file = "data/VGAE/filtered_generated_smiles.csv"
    print("=" * 80)
    print(f"Starting SMILES validation and enrichment script")
    print(f"Input file: {input_file}")
    print("=" * 80)
    results = process_smiles_file(input_file)
    print("=" * 80)
    print("Script execution completed")
    print("=" * 80)

Starting SMILES validation and enrichment script
Input file: data/VGAE/filtered_generated_smiles.csv
Reading file: data/VGAE/filtered_generated_smiles.csv
Successfully loaded file with 3 rows and 1 columns
Processing 3 molecules...
Processing molecule 1/3...
  Calculating properties for molecule 1
  Molecule 1 QED (not saved to CSV): 0.4310243576713091
  Calculating synthesizability for molecule 1
  Fetching PubChem data for molecule 1
  Fetching ChEMBL data for molecule 1
  Searching for patents for molecule 1
  Summary for molecule 1:
    Molecular weight: 58.12
    LogP: 1.81
    Synthetic accessibility score: 1.61 (Easy)
    Found in PubChem (CID: 7843.0)
    Found in ChEMBL (ID: CHEMBL134702)
  Calculating properties for molecule 2
  Molecule 2 QED (not saved to CSV): 0.34449676855385364
  Calculating synthesizability for molecule 2
  Fetching PubChem data for molecule 2
  Fetching ChEMBL data for molecule 2
  Searching for patents for molecule 2
  Summary for molecule 2:
    Mole

Possible Molecular Fragments: C2H6O


1.1592175456870955